# DArtP with PathBench

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karkirowle/pathbench/blob/main/examples/dartp_colab.ipynb)

This notebook shows how to score your own recordings with **DArtP**, the best reference-free metric in the [PathBench results table](https://github.com/karkirowle/pathbench/blob/main/results/results_table.csv). It is meant for programmers: it covers the Python API, not the benchmark.

DArtP needs only the audio, not a transcript. For each utterance it:

1. transcribes the audio with a language-specific wav2vec 2.0 ASR model and a KenLM n-gram language model, which gives the words the speaker most likely *intended*;
2. turns that transcript into phonemes with espeak-ng;
3. force-aligns those phonemes to the audio with a phonetic wav2vec 2.0 model (`facebook/wav2vec2-xlsr-53-espeak-cv-ft`);
4. returns the mean posterior probability of the aligned phonemes, between 0 and 1. **Higher means more precise articulation.**

**Runtime.** It runs on the free tier, CPU or GPU. The language model is loaded lazily and the whole pipeline peaks at about 4 GB of RAM. The first run downloads about 8.6 GB (the English LM) plus about 2.5 GB of models.

## 1. Install

espeak-ng is a system library that pip cannot install. The paper uses espeak-ng commit `2ea41210`, while Colab's `apt` package is an older release. For English the difference is usually negligible, but scores can differ slightly from the paper. See [Installation](https://github.com/karkirowle/pathbench#installation) for the exact route.

The `[lm]` extra installs `kenlm`, which compiles from source. This takes a minute or two.

In [ ]:
!apt-get -qq install -y espeak-ng > /dev/null
!espeak-ng --version
!pip install -q "pathbench[lm] @ git+https://github.com/karkirowle/pathbench.git"

## 2. Download the English language model

The n-gram models are published in `lms.zip` on Zenodo ([record 18738598](https://zenodo.org/records/18738598), CC BY 4.0). The zip holds all languages (35 GB). The code below reads only the zip's table of contents, then streams just the English model (8.6 GB compressed, 14.6 GB on disk) with a single HTTP range request, decompressing on the fly and checking its CRC.

This takes a while. If the download is interrupted, re-run the cell; it starts over. Once the file exists, the cell skips the download.

In [ ]:
import io
import os
import struct
import zipfile
import zlib

import requests

LMS_ZIP_URL = "https://zenodo.org/records/18738598/files/lms.zip?download=1"
LM_MEMBER = "lms/wiki_en_token.arpa.bin"
LM_PATH = "/content/lms/wiki_en_token.arpa.bin"


class HTTPRangeFile(io.RawIOBase):
    """Read-only, seekable view of a remote file, fetched with HTTP range requests."""

    def __init__(self, url):
        self.url, self.pos = url, 0
        self.size = int(requests.head(url, allow_redirects=True).headers["Content-Length"])

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=io.SEEK_SET):
        self.pos = {io.SEEK_SET: offset, io.SEEK_CUR: self.pos + offset, io.SEEK_END: self.size + offset}[whence]
        return self.pos

    def readinto(self, buffer):
        if self.pos >= self.size or len(buffer) == 0:
            return 0
        end = min(self.pos + len(buffer), self.size) - 1
        r = requests.get(self.url, headers={"Range": f"bytes={self.pos}-{end}"})
        r.raise_for_status()
        buffer[:len(r.content)] = r.content
        self.pos += len(r.content)
        return len(r.content)


def download_zip_member(url, member, dest, chunk_size=1 << 20):
    """Extract one deflate-compressed member of a remote zip without downloading the rest."""
    with zipfile.ZipFile(io.BufferedReader(HTTPRangeFile(url), 1 << 16)) as z:
        info = z.getinfo(member)
    if os.path.exists(dest) and os.path.getsize(dest) == info.file_size:
        print(f"{dest} already present, skipping download.")
        return dest
    assert info.compress_type == zipfile.ZIP_DEFLATED, info.compress_type

    # The compressed data starts after the member's local header (30 bytes + name + extra).
    header = requests.get(url, headers={"Range": f"bytes={info.header_offset}-{info.header_offset + 29}"}).content
    name_len, extra_len = struct.unpack("<HH", header[26:30])
    start = info.header_offset + 30 + name_len + extra_len
    end = start + info.compress_size - 1

    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp = dest + ".part"
    inflater, crc, written, next_report = zlib.decompressobj(-zlib.MAX_WBITS), 0, 0, 0
    with requests.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True) as r, open(tmp, "wb") as f:
        r.raise_for_status()
        for block in r.iter_content(chunk_size):
            data = inflater.decompress(block)
            f.write(data)
            crc, written = zlib.crc32(data, crc), written + len(data)
            if written >= next_report:
                print(f"\r{written / 1e9:5.1f} / {info.file_size / 1e9:.1f} GB", end="", flush=True)
                next_report += 500_000_000
        data = inflater.flush()
        f.write(data)
        crc, written = zlib.crc32(data, crc), written + len(data)
    if written != info.file_size or crc != info.CRC:
        raise IOError(f"{member}: size/CRC mismatch ({written} bytes, crc {crc:08x}); re-run to retry")
    os.replace(tmp, dest)
    print(f"\nSaved {dest}")
    return dest


download_zip_member(LMS_ZIP_URL, LM_MEMBER, LM_PATH)

## 3. Create the evaluator

`lm_path` points DArtP at the downloaded model. Without it, the evaluator looks for `lms/` in the current directory, the layout of the benchmark repository. Use espeak-ng language codes (`en-us`, not `en`).

The first call downloads the two wav2vec 2.0 models from Hugging Face. They are pinned to the revisions used for the paper.

In [ ]:
from pathbench import ArtPDoubleASREvaluator

dartp = ArtPDoubleASREvaluator(language="en-us", lm_path=LM_PATH)

## 4. Score an example recording

The example is a sentence of clear, typical read speech from [LibriTTS](https://www.openslr.org/60/) (utterance `4830_25904_000004_000003`, CC BY 4.0; see [`examples/audio/README.md`](https://github.com/karkirowle/pathbench/blob/main/examples/audio/README.md)): *"So I threw him some fresh clover, and went on to the pelicans."*

`score(utterance_id, audio_path)` loads any audio format librosa reads and resamples it to 16 kHz. It also prints the phonemes it aligned.

In [ ]:
import urllib.request

import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/karkirowle/pathbench/main/"
sentence = "libritts_4830_25904_000004_000003.wav"
urllib.request.urlretrieve(REPO_RAW + "examples/audio/" + sentence, sentence)

dartp.score("libritts", sentence)

### What did DArtP hear?

DArtP scores the speaker against the words its ASR step *recognised*, so the transcript is the first thing to check when a score looks off. The helper below prints it next to the score. It uses the evaluator's internal ASR model and decoder.

The second clip shows a typical failure. It is a 0.18 s recording of the single word *blue* from PathBench's unit tests (CC0). That is too little audio for the ASR to recognise anything, so DArtP falls back to aligning a single dummy phoneme and returns a meaningless score close to 0. Score sentences, or at least several words, and average over utterances, as the benchmark does.

In [ ]:
import librosa
import torch


def dartp_with_transcript(path):
    audio, fs = librosa.load(path, sr=16000)
    inputs = dartp.processor(audio, sampling_rate=fs, return_tensors="pt").input_values.to(dartp.device)
    with torch.no_grad():
        logits = dartp.model(inputs).logits.cpu().numpy()[0]
    return {"file": path, "ASR transcript": dartp.decoder.decode(logits), "DArtP": dartp.score(path, path)}


too_short = "BLUE_english34.wav"
urllib.request.urlretrieve(REPO_RAW + "tests/data/" + too_short, too_short)
pd.DataFrame([dartp_with_transcript(sentence), dartp_with_transcript(too_short)])

## 5. Score your own audio

Upload one or more files (wav, flac, mp3, ...), ideally one sentence per file. `score()` returns `None` when it cannot score a file, e.g. audio shorter than 25 ms.

In [ ]:
from google.colab import files

uploaded = files.upload()
pd.DataFrame([dartp_with_transcript(name) for name in uploaded])

## 6. Scoring a whole set of speakers

In PathBench a speaker's score is the mean of their utterance scores. These speaker scores are then correlated with clinical severity ratings. A minimal version for your own data looks like this: one row per utterance, then a per-speaker mean.

To trim an utterance, pass `start_time` and `end_time` in seconds: `dartp.score(utt_id, path, start_time=0.5, end_time=2.0)`.

In [ ]:
# Replace with your own (speaker, file) pairs, e.g.
# utterances = [("speaker_a", "a_001.wav"), ("speaker_a", "a_002.wav"), ("speaker_b", "b_001.wav")]
utterances = [("libritts_4830", sentence)]

scores = pd.DataFrame(
    [{"speaker": spk, "file": path, "DArtP": dartp.score(path, path)} for spk, path in utterances]
)
scores.groupby("speaker")["DArtP"].mean()

## Other languages

`lms.zip` also contains models for Spanish (`lms/wiki_es_token.arpa.bin`), Italian (`lms/wiki_it_token.arpa.bin`) and Dutch (`lms/wiki_nl_token.arpa.bin`). Download one with `download_zip_member` and create the evaluator with `language="es"`, `"it"` or `"nl"`. The Mandarin model used in the paper is not part of `lms.zip`.

For the full benchmark (datasets, all metrics, speaker-level correlations) see the [repository README](https://github.com/karkirowle/pathbench).